# Posterior distortion, bound usefulness and the target
[Proof](../12_posterior_precision_distortion.md). Original Gaussian checks are retained; new diagnostics do not count as learned-model evidence.

In [ ]:
import numpy as np
from experiments.sampled_conditioning.core import gaussian_kl, precision_certificate
from experiments.sampled_conditioning.parameterization_controls import bounds, denoiser_gap
rng=np.random.default_rng(12)
errors=[]; ratios=[]; rows=[]
for i in range(100):
    a,b=rng.normal(size=(4,4)),rng.normal(size=(4,4))
    p,q=a@a.T+np.eye(4),b@b.T+np.eye(4)
    eta,etq=rng.normal(size=4),rng.normal(size=4)
    result=precision_certificate(p,eta,q,etq)
    direct=gaussian_kl(np.linalg.solve(p,eta),np.linalg.inv(p),np.linalg.solve(q,etq),np.linalg.inv(q))
    tight=bounds(p,eta,q,etq)
    errors.append(abs(direct-result['kl_nats']))
    assert direct<=tight['tight_bound']+1e-8<=tight['old_bound']+2e-8
    np.testing.assert_allclose(result['bound_nats'],tight['old_bound'],rtol=1e-8,atol=1e-8)
    rows.append([i,direct,tight['old_bound'],tight['tight_bound'],tight['tight_bound']/direct])
assert max(errors)<1e-9
rows=np.array(rows)
print('max independent KL discrepancy',max(errors))
print('columns: example, exact KL, old bound, tight bound, tight/exact')
print(rows[:8])
print('tight/exact min, median, max',np.min(rows[:,4]),np.median(rows[:,4]),np.max(rows[:,4]))
print('minimum exact and minimum bound example',int(rows[np.argmin(rows[:,1]),0]),int(rows[np.argmin(rows[:,3]),0]))

## Joint error and a zero perturbation
Precision and natural-parameter errors can cancel; use their interaction, not two independent norm rankings.

In [ ]:
p=np.eye(2);q=np.array([[1.,.4],[.4,1.]])
values=[precision_certificate(p,np.array([s,0.]),q,np.array([s,0.]))['kl_nats'] for s in [0.,1.,10.]]
assert values[2]>values[1]>values[0]
zero=bounds(p,np.ones(2),p,np.ones(2))
assert zero['exact_kl']==zero['tight_bound']==0
h=np.array([1.,2.]);e=q-p
cancel=bounds(p,h,q,h+e@h);amplify=bounds(p,h,q,h-e@h)
assert cancel['mean_residual_squared']<1e-25
assert amplify['exact_kl']>cancel['exact_kl']
print('scale test',values,'cancel',cancel,'amplify',amplify)

## Fixed-target Gaussian denoiser discrepancy
The target is already projected to two dimensions. Both predictors receive the same noisy target drawn from the true law.

In [ ]:
mu=np.array([.2,-.3]);mq=np.array([-.1,.4])
v=np.array([[.8,.2],[.2,.5]]);w=np.diag([.6,1.2]);a,s=.8,.6
m=a*a*v+s*s*np.eye(2);n=a*a*w+s*s*np.eye(2)
z=np.random.default_rng(2).multivariate_normal(a*mu,m,size=100000)
f=s*np.linalg.solve(m,(z-a*mu).T).T
fq=s*np.linalg.solve(n,(z-a*mq).T).T
losses=np.sum((f-fq)**2,axis=1);expected=denoiser_gap(mu,v,mq,w,a,s)
se=losses.std(ddof=1)/np.sqrt(len(losses))
assert abs(losses.mean()-expected)<5*se
print('Monte Carlo, exact discrepancy, SE',losses.mean(),expected,se)
assert denoiser_gap(mu,v,mq,w,0.,1.)==0
# Joint ranking does not order the goal-projected ranking.
p=np.zeros(2);a=np.array([0.,.9]);b=np.array([1.,0.]);eye=np.eye(2);L=np.array([[0.,1.]])
assert gaussian_kl(p,eye,a,eye)<gaussian_kl(p,eye,b,eye)
assert gaussian_kl(L@p,L@eye@L.T,L@a,L@eye@L.T)>gaussian_kl(L@p,L@eye@L.T,L@b,L@eye@L.T)

The bound is an offline oracle diagnostic and can be loose or rank candidates differently. The target formula is a Gaussian plug-in discrepancy, not a proof that a learned LLapDiff improves.

In [ ]:
print("THEORY_DEMO_PASS::12_posterior_precision_distortion")